In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv('/content/URL_Labels.csv')

def classify_url(url: str):
    path = url.replace('https://www.theguardian.com/', '').strip('/')
    parts = path.split('/')
    if not parts:
        return None, None, False  # URL không hợp lệ

    category = parts[0]          # cấp 1 luôn có
    sub_section = None
    is_level2 = False

    # Nếu phần tiếp theo không phải là năm (4 chữ số) → đó là sub-section
    if len(parts) > 1 and not re.fullmatch(r'\d{4}', parts[1]):
        sub_section = parts[1]
        is_level2 = True

    return category, sub_section, is_level2

# Áp dụng hàm phân loại
df[['category', 'sub_section', 'is_level2']] = df['url'].apply(
    lambda x: pd.Series(classify_url(x))
)

# Tổng số URL
total = len(df)
cap1_count = (~df['is_level2']).sum()   # số URL cấp 1
cap2_count = df['is_level2'].sum()      # số URL cấp 2

print(f'Tổng số URL: {total}')
print(f'URL cấp 1 (category + năm): {cap1_count}')
print(f'URL cấp 2 (category + sub-section + năm): {cap2_count}')
print()

Tổng số URL: 41919
URL cấp 1 (category + năm): 39637
URL cấp 2 (category + sub-section + năm): 2282



In [3]:
df.loc[~df['is_level2'], 'category'].value_counts()

,count
category,
film,5995
sport,5336
music,5241
culture,3977
food,3845
world,2548
environment,2256
money,2157
business,2075


In [4]:
df.loc[df['is_level2'], 'sub_section'].value_counts()

,count
sub_section,
live,1286
blog,391
shortcuts,205
nils-pratley-on-finance,200
commentisfree,139
grogonomics,30
filmblog,11
costume-and-culture,6
from-the-archive-blog,4


In [5]:
# LỌC BỎ CẤP 2, GIỮ LẠI CẤP 1
df_level1 = df[~df['is_level2']].copy()

# In thông tin
print(f"Tổng số URL ban đầu: {len(df)}")
print(f"Số URL cấp 1 giữ lại: {len(df_level1)}")
print(f"Số URL cấp 2 đã loại bỏ: {len(df) - len(df_level1)}")

Tổng số URL ban đầu: 41919
Số URL cấp 1 giữ lại: 39637
Số URL cấp 2 đã loại bỏ: 2282


In [6]:
df_level1.to_csv('URL_Level1_only.csv', encoding='utf-8', index=False)